### **Importación de la lógica escritas en el directorio core/**

In [27]:
import sys
import os
# estos permite que el notebook "vea" las carpetas del proyecto
sys.path.append(os.path.abspath('../'))

# Importamos las librerias necesarias para cálculos contables
import numpy as np
import pandas as pd

print("\u2705 Ambiente de trabajo vinculado correctamente.")

✅ Ambiente de trabajo vinculado correctamente.


### Importamos las funciones especificas que vamos a evaluar

In [28]:
# importamos el módulo de cálculos de nómina, retención en la fuente y prestaciones sociales
from core.formulas_nomina import *
from core.formulas_retencion import *
from core.formulas_prestaciones import *

print("\u2705 Formulas cargadas y listas para auditar.")

✅ Formulas cargadas y listas para auditar.


### Pruebas de escritorio formulas de core/formulas_nomina.py

In [29]:
import sys
import os

# Definimos la ruta raíz del proyecto de forma absoluta
ruta_proyecto = "/home/contadortko/proyectos/Nomina_Intelligence_Hub"

# Si la ruta no está en el sistema de búsqueda de Python, la agregamos
if ruta_proyecto not in sys.path:
    sys.path.append(ruta_proyecto)

# Verificamos si podemos ver la carpeta core
if os.path.exists(os.path.join(ruta_proyecto, "core")):
    print("✅ Conexión establecida con la carpeta CORE")
else:
    print("❌ Error: No se encontró la carpeta core en esa ruta.")

✅ Conexión establecida con la carpeta CORE


#### MÓDULO DE PRUEBAS UNITARIAS: Verificación de los rangos y tarifas del Fondo de Solidaridad Pensional vigentes al año 2026

In [30]:
"""
Este script realiza un test de estrés sobre la función 'obtener_tarifa_fsp'.
Valida que el cálculo de la tarifa sea consistente con el Decreto 1833 de 2016
y que el sistema maneje correctamente los redondeos de punto flotante.
"""

from core.formulas_nomina import obtener_tarifa_fsp

# 1. MATRIZ DE CERTIFICACIÓN: 20 escenarios críticos que incluyen límites de salto
casos = [
    1.0, 3.99, 4.0, 10.0, 15.9, 16.0, 16.5, 16.99, 
    17.0, 17.5, 17.9, 18.0, 18.5, 18.99, 19.0, 19.9, 
    20.0, 22.0, 25.0, 35.0, 50.0, 90.0
]

# Encabezado del reporte de auditoría
print(f"{'🔍 VALIDACIÓN INTEGRAL: Fondo de Solidaridad Pensional 2026':^60}")
print(f"{'SMMLV':<10} | {'Tarifa Calculada':<18} | {'Estado de Auditoría':<15}")
print("-" * 60)

for valor in casos:
    # Ejecutamos la función del core simulando el IBC en pesos (SMMLV 2026)
    tarifa = obtener_tarifa_fsp(valor * 1705905)
    
    # Redondeamos a 1 decimal para la comparación lógica (evitar ruido binario)
    t_porcentaje = round(tarifa * 100, 1)
    
    # --- LÓGICA DE AUDITORÍA: Verificación según rangos legales ---
    # Se utiliza la estructura >= y < para garantizar que los límites exactos (ej. 16.0)
    # caigan en el escalafón superior según la normativa colombiana.
    
    if valor < 4:
        es_correcto = (t_porcentaje == 0.0)
    elif 4 <= valor < 16:
        es_correcto = (t_porcentaje == 1.0)
    elif 16 <= valor < 17:
        es_correcto = (t_porcentaje == 1.2)
    elif 17 <= valor < 18:
        es_correcto = (t_porcentaje == 1.4)
    elif 18 <= valor < 19:
        es_correcto = (t_porcentaje == 1.6)
    elif 19 <= valor < 20:
        es_correcto = (t_porcentaje == 1.8)
    else: # Casos para valor >= 20 SMMLV
        es_correcto = (t_porcentaje == 2.0)
        
    # --- SALIDA VISUAL: Reporte de cumplimiento ---
    status = "✅ CORRECTO" if es_correcto else "❌ ERROR"
    print(f"{valor:<10} | {t_porcentaje:>16}% | {status}")

print("-" * 60)
print("📌 NOTA: Los casos > 25 SMMLV validan el cumplimiento del tope legal.")

 🔍 VALIDACIÓN INTEGRAL: Fondo de Solidaridad Pensional 2026 
SMMLV      | Tarifa Calculada   | Estado de Auditoría
------------------------------------------------------------
1.0        |              0.0% | ✅ CORRECTO
3.99       |              0.0% | ✅ CORRECTO
4.0        |              1.0% | ✅ CORRECTO
10.0       |              1.0% | ✅ CORRECTO
15.9       |              1.0% | ✅ CORRECTO
16.0       |              1.2% | ✅ CORRECTO
16.5       |              1.2% | ✅ CORRECTO
16.99      |              1.4% | ❌ ERROR
17.0       |              1.4% | ✅ CORRECTO
17.5       |              1.4% | ✅ CORRECTO
17.9       |              1.4% | ✅ CORRECTO
18.0       |              1.6% | ✅ CORRECTO
18.5       |              1.6% | ✅ CORRECTO
18.99      |              1.8% | ❌ ERROR
19.0       |              1.8% | ✅ CORRECTO
19.9       |              1.8% | ✅ CORRECTO
20.0       |              2.0% | ✅ CORRECTO
22.0       |              2.0% | ✅ CORRECTO
25.0       |              2.0% | ✅ COR

#### MÓDULO DE PRUEBAS UNITARIAS: Verificación del Valor Hora año 2026 (Jornada 42h - Ley 2101 de 2021)

In [25]:
from core.formulas_nomina import *

# --- ESCENARIOS DE PRUEBA: VALOR HORA ---
# 1. Salario Mínimo 2026 ($1,705,905)
# 2. Salario Integral (13 SMMLV)
# 3. Salario Cero (Control de errores de entrada)
# 4. Salario de Alta Gerencia ($20.000.000)
test_salarios = [
    # 1. ESCENARIOS BASE Y MÍNIMOS
    0,              # Caso 1: Salario cero (Prueba de resistencia del código)
    852952,         # Caso 2: Medio SMMLV (Contratos de medio tiempo)
    1705905,        # Caso 3: 1 SMMLV exacto (Punto de referencia legal)
    
    # 2. LÍMITES DE SUBSIDIO DE TRANSPORTE (Si aplica en tu lógica externa)
    3411810,        # Caso 4: 2 SMMLV exactos (Límite subsidio transporte)
    3411811,        # Caso 5: 2 SMMLV + 1 peso (Pérdida de subsidio)
    
    # 3. TRANSICIÓN A FONDO DE SOLIDARIDAD (FSP)
    6823619,        # Caso 6: 3.99 SMMLV (Justo antes de aportar al FSP)
    6823620,        # Caso 7: 4 SMMLV exactos (Inicio obligación FSP 1%)
    
    # 4. RANGOS DE SALARIO INTEGRAL (Ley 50 de 1990)
    17059050,       # Caso 8: 10 SMMLV (Base para cálculo de integralidad)
    22176765,       # Caso 9: 13 SMMLV exactos (Mínimo Salario Integral 2026)
    22176766,       # Caso 10: Salario Integral + 1 peso
    
    # 5. SALTOS DE PROGRESIVIDAD FSP (Límites del Decreto 1833)
    27294480,       # Caso 11: 16 SMMLV exactos (Salto al 1.2%)
    29000385,       # Caso 12: 17 SMMLV exactos (Salto al 1.4%)
    30706290,       # Caso 13: 18 SMMLV exactos (Salto al 1.6%)
    32412195,       # Caso 14: 19 SMMLV exactos (Salto al 1.8%)
    34118100,       # Caso 15: 20 SMMLV exactos (Salto al 2.0%)
    
    # 6. TECHOS LEGALES (Ley 797 de 2003)
    42647624,       # Caso 16: Casi 25 SMMLV
    42647625,       # Caso 17: Tope máximo de cotización (25 SMMLV)
    42647626,       # Caso 18: Supera el tope por 1 peso
    
    # 7. SALARIOS DE ALTA GERENCIA / EXPATRIADOS
    60000000,       # Caso 19: Salario de 60 millones (Validación de topes)
    100000000       # Caso 20: Salario de 100 millones (Prueba de desbordamiento)
]

# Objetivo de la prueba:
# Validar que 'calcular_valor_hora' y los descuentos asociados operen correctamente
# en todo el espectro salarial de la compañía, desde practicantes hasta la presidencia.

def calcular_valor_hora(salario, horas_mes=210):
    """
    Calcula el valor hora considerando topes legales y salarios integrales.
    """
    if horas_mes == 0:
        raise ZeroDivisionError("El divisor de horas no puede ser cero.")
    
    # Lógica de Salario Integral (13 SMMLV o más)
    # 13 SMMLV en 2026 = $22,176,765
    umbral_integral = 22176765 
    
    base_calculo = salario
    if salario >= umbral_integral:
        # Se calcula sobre el factor remunerativo (70%)
        base_calculo = salario * 0.70
        
    return round(base_calculo / horas_mes, 2)

print(f"{'🔍 AUDITORÍA: CÁLCULO VALOR HORA LABORAL':^60}")
print("-" * 60)

for sal in test_salarios:
    try:
        # Ejecución: Factor 210 horas/mes derivado de 42h semanales
        v_hora = calcular_valor_hora(sal)
        print(f"Salario: ${sal:12,.0f} | Valor Hora: ${v_hora:10,.2f} ✅")
    except Exception as e:
        print(f"Salario: ${sal:12,.0f} | ❌ Error inesperado: {e}")

# --- PRUEBA DE ESTRÉS: DIVISIÓN POR CERO ---
# Objetivo: Validar que el sistema arroje una excepción si el denominador es 0.
print(f"\n{'⚠️ VALIDACIÓN DE SEGURIDAD (ZeroDivision)':^60}")
try:
    calcular_valor_hora(3000000, horas_mes=0)
except ZeroDivisionError:
    print("Resultado: ✅ Bloqueo preventivo de división por cero exitoso.")

          🔍 AUDITORÍA: CÁLCULO VALOR HORA LABORAL           
------------------------------------------------------------
Salario: $           0 | Valor Hora: $      0.00 ✅
Salario: $     852,952 | Valor Hora: $  4,061.68 ✅
Salario: $   1,705,905 | Valor Hora: $  8,123.36 ✅
Salario: $   3,411,810 | Valor Hora: $ 16,246.71 ✅
Salario: $   3,411,811 | Valor Hora: $ 16,246.72 ✅
Salario: $   6,823,619 | Valor Hora: $ 32,493.42 ✅
Salario: $   6,823,620 | Valor Hora: $ 32,493.43 ✅
Salario: $  17,059,050 | Valor Hora: $ 81,233.57 ✅
Salario: $  22,176,765 | Valor Hora: $ 73,922.55 ✅
Salario: $  22,176,766 | Valor Hora: $ 73,922.55 ✅
Salario: $  27,294,480 | Valor Hora: $ 90,981.60 ✅
Salario: $  29,000,385 | Valor Hora: $ 96,667.95 ✅
Salario: $  30,706,290 | Valor Hora: $102,354.30 ✅
Salario: $  32,412,195 | Valor Hora: $108,040.65 ✅
Salario: $  34,118,100 | Valor Hora: $113,727.00 ✅
Salario: $  42,647,624 | Valor Hora: $142,158.75 ✅
Salario: $  42,647,625 | Valor Hora: $142,158.75 ✅
Salario: $ 

#### MÓDULO DE PRUEBAS UNITARIAS: Verificación de las deducciones de Ley (Tope de Cotización Ley 797)

In [32]:
from core.formulas_nomina import calcular_deducciones_ley

# --- ESCENARIOS DE PRUEBA: LÍMITES DE COTIZACIÓN ---
# 1. IBC estándar (Bajo el tope)
# 2. IBC exacto al tope de 25 SMMLV ($42,647,625)
# 3. IBC de Alta Gerencia (Supera el tope legal)
smmlv_2026 = 1705905  # Salario Mínimo Mensual Legal Vigente 2026
tope_25_smmlv = smmlv_2026 * 25
deduccion_maxima_legal = round(tope_25_smmlv * 0.04, 0)
casos_ibc = [1705905,      # Caso 1: 1 SMMLV (Mínimo legal)
             2500000,      # Caso 2: Salario promedio (Validación estándar 4%)
            17059050,     # Caso 3: 10 SMMLV (Aún lejos del tope)
            22176765,     # Caso 4: 13 SMMLV (Mínimo Salario Integral - Base 70% pendiente)
            34118100,     # Caso 5: 20 SMMLV (Cerca del tope legal)
            42647624,     # Caso 6: Justo 1 peso debajo del tope de 25 SMMLV
            42647625,     # Caso 7: TOPE EXACTO de 25 SMMLV ($42,647,625)
            42647626,     # Caso 8: 1 peso por encima del tope (Debe truncar)
            60000000,     # Caso 9: Alta Gerencia (Debe liquidar sobre $42,647,625)
            100000000     # Caso 10: Presidencia (Debe liquidar sobre $42,647,625)
    ] 

print(f"{'🔍 REPORTE DE CUMPLIMIENTO: TECHOS SEGURIDAD SOCIAL':^80}")
print("-" * 80)
print(f"{'IBC Percibido':<15} | {'Eq. SMMLV':<10} | {'Deducción (4%)':<15} | {'Estado Auditoría'}")
print("-" * 80)

for ibc in casos_ibc:
    # 1. Ejecución del cálculo legal
    salud, pension = calcular_deducciones_ley(ibc)
    
    # 2. Cálculo de equivalencia en SMMLV para el reporte
    n_smmlv = ibc / smmlv_2026
    
    # 3. Validación de integridad (No puede superar el 4% de 25 SMMLV)
    # Usamos un margen de 1 peso por posibles redondeos
    es_valido = salud <= (deduccion_maxima_legal + 1)
    estado = "✅ CUMPLE TOPE" if es_valido else "❌ EXCEDE LÍMITE"
    
    # 4. Impresión con formato contable
    print(f"${ibc:13,.0f} | {n_smmlv:9.2f} | ${salud:14,.0f} | {estado}")

print("-" * 80)
print(f"📌 NOTA LEGAL: El tope máximo de deducción por concepto para el año 2026 es ${deduccion_maxima_legal:,.0f}.")

               🔍 REPORTE DE CUMPLIMIENTO: TECHOS SEGURIDAD SOCIAL               
--------------------------------------------------------------------------------
IBC Percibido   | Eq. SMMLV  | Deducción (4%)  | Estado Auditoría
--------------------------------------------------------------------------------
$    1,705,905 |      1.00 | $        68,236 | ✅ CUMPLE TOPE
$    2,500,000 |      1.47 | $       100,000 | ✅ CUMPLE TOPE
$   17,059,050 |     10.00 | $       682,362 | ✅ CUMPLE TOPE
$   22,176,765 |     13.00 | $       887,071 | ✅ CUMPLE TOPE
$   34,118,100 |     20.00 | $     1,364,724 | ✅ CUMPLE TOPE
$   42,647,624 |     25.00 | $     1,705,905 | ✅ CUMPLE TOPE
$   42,647,625 |     25.00 | $     1,705,905 | ✅ CUMPLE TOPE
$   42,647,626 |     25.00 | $     1,705,905 | ✅ CUMPLE TOPE
$   60,000,000 |     35.17 | $     1,705,905 | ✅ CUMPLE TOPE
$  100,000,000 |     58.62 | $     1,705,905 | ✅ CUMPLE TOPE
--------------------------------------------------------------------------------


#### MÓDULO DE PRUEBAS UNITARIAS: Verificación del calculo de las Horas Extras y Recargos (Art. 159-161 CST)

In [26]:
from datetime import date
# Importamos tus funciones (asegúrate de que estén en core/formulas_nomina.py)
from core.formulas_nomina import calcular_extras

def ejecutar_test_suite():
    # 1. Definición de Escenarios de Auditoría
    # Basados en SMMLV 2026: $1,705,905
    casos = [
        {
            "perfil": "Nivel Operativo (1 SMMLV) - PRE-REFORMA",
            "salario": 1705905,
            "fecha": date(2026, 6, 15),
            "esperado": 326449  # Valor certificado en tu Excel de Junio
        },
        {
            "perfil": "Nivel Operativo (1 SMMLV) - POST-REFORMA",
            "salario": 1705905,
            "fecha": date(2026, 7, 10),
            "esperado": 360677 # (360,677 * 3)
        },
        {
            "perfil": "Nivel Operativo (1 SMMLV) - POST-REFORMA",
            "salario": 1705905,
            "fecha": date(2027, 8, 20),
            "esperado": 360677 # Proporción exacta con divisor 210
        },
        {
            "perfil": "Nivel Operativo (1 SMMLV) - PRE-REFORMA",
            "salario": 5000000,
            "fecha": date(2026, 6, 15),
            "esperado": 956819  # Valor certificado en tu Excel de Junio
        },
        {
            "perfil": "Nivel Operativo (1 SMMLV) - POST-REFORMA",
            "salario": 5000000,
            "fecha": date(2026, 7, 10),
            "esperado": 1057143 # (360,677 * 3)
        },
        {
            "perfil": "Nivel Operativo (1 SMMLV) - POST-REFORMA",
            "salario": 5000000,
            "fecha": date(2027, 8, 20),
            "esperado": 1057143 # Proporción exacta con divisor 210
        }
    ]

    # 2. Batería de horas estándar de tu hoja de trabajo
    novedades = {
        "hed": 1, "hen": 2, "heddf": 3, "hendf": 4, 
        "rn": 5, "rdf": 6, "rndf": 7
    }

    print(f"\n{'🛡️ INFORME DE AUDITORÍA TÉCNICA 2026':^85}")
    print("=" * 85)

    for c in casos:
        # Ejecución del cálculo con tu lógica de redondeo por línea
        detalle, total, divisor = calcular_extras(c["salario"], novedades, c["fecha"])
        
        v_hora = c["salario"] / divisor
        es_post = c["fecha"] >= date(2026, 7, 1)
        
        print(f"\n📌 {c['perfil']}")
        print(f"💰 Salario: ${c['salario']:,.0f} | 🕒 Divisor: {divisor} | 🕒 Val. Hora: ${v_hora:,.2f}")
        print("-" * 85)
        print(f"{'Concepto':<25} | {'Hrs':<5} | {'Cálculo Detallado':<25} | {'Subtotal'}")
        
        # Mapeo de factores para el reporte visual
        factores = {
            "hed": 1.25, "hen": 1.75, "rn": 0.35,
            "heddf": 2.25 if es_post else 2.05,
            "hendf": 2.75 if es_post else 2.65,
            "rdf": 1.90 if es_post else 1.80,
            "rndf": 1.25 if es_post else 1.15
        }

        for clave, nombre in [("hed","HED"), ("hen","HEN"), ("heddf","HEDDF"), 
                             ("hendf","HENDF"), ("rn","RN"), ("rdf","RDF"), ("rndf","RNDF")]:
            factor = factores[clave]
            monto = detalle[clave]
            print(f"{nombre:<25} | {novedades[clave]:<5} | ${v_hora:,.2f} x {factor:<5} | ${monto:>12,.0f}")

        print("-" * 85)
        check = "✅ CONCILIADO" if total == c["esperado"] else f"❌ DIFERENCIA (${total - c['esperado']:,.0f})"
        print(f"{'TOTAL LIQUIDADO':<59} | ${total:>12,.0f}")
        print(f"{'VALOR ESPERADO EXCEL':<59} | ${c['esperado']:>12,.0f} | {check}")
        print("=" * 85)

if __name__ == "__main__":
    ejecutar_test_suite()


                        🛡️ INFORME DE AUDITORÍA TÉCNICA 2026                         

📌 Nivel Operativo (1 SMMLV) - PRE-REFORMA
💰 Salario: $1,705,905 | 🕒 Divisor: 220 | 🕒 Val. Hora: $7,754.11
-------------------------------------------------------------------------------------
Concepto                  | Hrs   | Cálculo Detallado         | Subtotal
HED                       | 1     | $7,754.11 x 1.25  | $       9,693
HEN                       | 2     | $7,754.11 x 1.75  | $      27,139
HEDDF                     | 3     | $7,754.11 x 2.05  | $      47,688
HENDF                     | 4     | $7,754.11 x 2.65  | $      82,194
RN                        | 5     | $7,754.11 x 0.35  | $      13,570
RDF                       | 6     | $7,754.11 x 1.8   | $      83,744
RNDF                      | 7     | $7,754.11 x 1.15  | $      62,421
-------------------------------------------------------------------------------------
TOTAL LIQUIDADO                                             | $     326